# 모듈을 사용하는 작업 시작 전 평가

함수는 risk_engine.py에서 관리합니다. 기존 03번은 보존했습니다.

먼저 화재감시자 예제와 6개 검사를 실행하세요. 전체 점수 계산에는 누락된 data/09_사전평가설정.csv가 필요합니다. 마지막 셀은 파일이 없으면 계산을 건너뛰고 안내합니다.


In [6]:
from pathlib import Path
import sys
import pandas as pd
candidates = [
    candidate
    for parent in [Path.cwd(), *Path.cwd().parents]
    for candidate in [parent, parent / "fire_risk_project"]
    if (candidate / "risk_engine.py").is_file()
]
if not candidates:
    raise FileNotFoundError("fire_risk_project 또는 notebooks 폴더에서 실행하세요.")
project_dir = candidates[0]
sys.path.insert(0, str(project_dir))
import risk_engine as engine
print("사용하는 계산 모듈:", engine.__file__)


사용하는 계산 모듈: c:\Users\uijun\OneDrive\바탕 화면\Project\fire_risk_project\risk_engine.py


In [7]:
with pd.option_context('display.max_colwidth', 100):
    display(engine.watcher_questions)


,추가질문ID,구분,질문,허용응답
0,W001,장소조건,작업반경 11m 이내 건물구조 자체 또는 내부(개구부 등으로 개방된 부분 포함)에 가연성물질이 있는가?,예 / 아니오 / 확인불가
1,W002,장소조건,작업반경 11m 이내의 바닥 하부에 가연성물질이 11m 이상 떨어져 있지만 불꽃에 의해 쉽게 발화될 우려가 있는가?,예 / 아니오 / 확인불가
2,W003,장소조건,금속 칸막이·벽·천장·지붕의 반대쪽 면에 가연성물질이 인접하여 열전도·열복사로 발화될 우려가 있는가?,예 / 아니오 / 확인불가
3,W004,예외검토,같은 장소에서 상시·반복적으로 용접·용단하는 경우의 설비 구비 등 법령상 배치 예외를 적용하려는가?,예 / 아니오 / 확인불가


In [8]:
watcher_answers = {
    "W001": "예", "W002": "아니오",
    "W003": "아니오", "W004": "아니오",
}
deployment_answer = "미배치"  # 가상 예제
watcher_check, watcher_detail = engine.check_h05_watcher_response(
    watcher_answers, deployment_answer
)
display(pd.DataFrame(watcher_check.items(), columns=["항목", "결과"]))


,항목,결과
0,작업ID,H05
1,대상항목ID,Q004
2,연결기준ID,R002
3,장소조건검토,장소조건 해당 응답 있음
4,해당조건ID,W001
5,미확인조건ID,-
6,검토결과,"배치 대상 검토: 장소조건 해당, 예외 적용 요청 없음"
7,해당없음자동승인,False
8,Q004응답,미배치
9,응답비교결과,배치 상태 확인 필요


## 화재감시자 분기 검사
원본 노트북의 6개 사례를 그대로 실행합니다.


In [9]:
test_cases = [
    (
        "조건 해당 + 미배치",
        {"W001": "예", "W002": "아니오",
         "W003": "아니오", "W004": "아니오"},
        "미배치",
        "배치 상태 확인 필요",
    ),
    (
        "조건 해당 + 배치",
        {"W001": "예", "W002": "아니오",
         "W003": "아니오", "W004": "아니오"},
        "배치",
        "배치 응답 확인",
    ),
    (
        "현장 조건 미확인",
        {"W001": "확인불가", "W002": "아니오",
         "W003": "아니오", "W004": "아니오"},
        "미배치",
        "현장 조건 확인 필요",
    ),
    (
        "배치 예외 검토 요청",
        {"W001": "예", "W002": "아니오",
         "W003": "아니오", "W004": "예"},
        "미배치",
        "예외 적용 검토 필요",
    ),
    (
        "해당없음 자동 승인 방지",
        {"W001": "아니오", "W002": "아니오",
         "W003": "아니오", "W004": "아니오"},
        "해당없음",
        "해당없음 근거 검토 필요",
    ),
    (
        "모든 장소조건 비해당",
        {"W001": "아니오", "W002": "아니오",
         "W003": "아니오", "W004": "아니오"},
        "미배치",
        "적용 여부 근거 확인 필요",
    ),
]

test_results = []

for name, answers, deployment, expected in test_cases:
    result, _ = engine.check_h05_watcher_response(answers, deployment)
    actual = result["응답비교결과"]

    test_results.append({
        "테스트": name,
        "예상결과": expected,
        "실제결과": actual,
        "통과": actual == expected,
    })

test_table = pd.DataFrame(test_results)
display(test_table)

assert test_table["통과"].all(), "실패한 테스트의 결과를 확인하세요."
print(f"총 {len(test_table)}개 테스트 통과")

,테스트,예상결과,실제결과,통과
0,조건 해당 + 미배치,배치 상태 확인 필요,배치 상태 확인 필요,True
1,조건 해당 + 배치,배치 응답 확인,배치 응답 확인,True
2,현장 조건 미확인,현장 조건 확인 필요,현장 조건 확인 필요,True
3,배치 예외 검토 요청,예외 적용 검토 필요,예외 적용 검토 필요,True
4,해당없음 자동 승인 방지,해당없음 근거 검토 필요,해당없음 근거 검토 필요,True
5,모든 장소조건 비해당,적용 여부 근거 확인 필요,적용 여부 근거 확인 필요,True


총 6개 테스트 통과


## 전체 위험점수 계산
09번 CSV 복원 후 실행하세요. 모듈 수정 시 커널을 재시작하세요.


In [11]:
if not (project_dir / 'data/09_사전평가설정.csv').is_file():
    print('전체 점수 계산 보류: data/09_사전평가설정.csv를 복원하세요.')
else:
    engine.load_databases(project_dir / 'data')
    h05_answers = {
        "Q001": "있음",
        "Q002": 5,
        "Q003": "미실시",
        "Q004": "미배치",
        "Q018": "용접방화포 미사용",
        "Q019": "없음",
        "Q020": "없음",
        "Q025": "적정",
    }
    summary, detail = engine.calculate_task_risk("H05", h05_answers)
    display(pd.DataFrame([summary]))
    display(detail)
    display(engine.get_top_risks(detail))
    
    prework_question_table = engine.get_prework_questions("H05", h05_answers)
    display(prework_question_table[["항목ID", "확인항목", "적용상태", "제외사유"]])

DB의 기본 연결·중복·집계 설정 검사 통과
사전평가 설정 검사 통과


,작업ID,평가시점,적용질문수,제외질문수,입력질문수,집계항목수,총가중위험점수,최대가중점수,위험지수,기본위험등급(잠정),강제경고,최종판정,계산상태
0,H05,작업시작전,8,1,8,7,13.5,30.0,45.0,주의,미검증,보류: 법령·잔여 적용조건 미검증,사전 시험 계산


,항목ID,확인항목,사용자응답,단위,점수규칙ID,조건순번,점수,가중치,가중점수,최대가중점수,...,그룹집계방식,관련기준ID,적용조건,확정상태,평가시점,적용상태,제외사유,사전안내,응답상태,집계대표여부
0,Q001,주변 가연물 존재 여부,있음,-,S001,1,1.0,1.5,1.5,4.5,...,MAX_PER_TASK,R001,화기·불티/전기/열원 작업 공통,미확정,작업시작전,적용,,작업 시작 직전에 확인한 현장 상태로 응답합니다. 미확인 상태를 안전한 응답으로 추...,입력,True
1,Q002,가연물과 작업점 거리,5.0,m,S002,1,1.0,1.5,1.5,4.5,...,MAX_PER_TASK,R001;R002,화기·불티 작업,미확정,작업시작전,적용,,작업 시작 직전에 확인한 현장 상태로 응답합니다. 미확인 상태를 안전한 응답으로 추...,입력,True
2,Q003,불티 비산방지조치(용접방화포 등) 실시 여부,미실시,-,S003,1,3.0,1.5,4.5,4.5,...,MAX_PER_TASK,R001,화기·불티 작업,미확정,작업시작전,적용,,작업 시작 직전에 확인한 현장 상태로 응답합니다. 미확인 상태를 안전한 응답으로 추...,입력,True
3,Q004,화재감시자 배치 여부,미배치,-,S004,1,3.0,2.0,6.0,6.0,...,MAX_PER_TASK,R002,법령상 배치대상 조건 충족 시 적용,미확정,작업시작전,적용,,작업 시작 직전에 확인한 현장 상태로 응답합니다. 미확인 상태를 안전한 응답으로 추...,입력,True
4,Q018,용접방화포 사용 시 성능인증 제품 여부,용접방화포 미사용,-,S019,3,0.0,0.0,0.0,0.0,...,MAX_PER_TASK,R001,다른 적정 비산방지조치를 적용하는 경우,미확정,작업시작전,적용,,작업 시작 직전에 확인한 현장 상태로 응답합니다. 미확인 상태를 안전한 응답으로 추...,입력,False
5,Q019,작업점 하부 가연물의 불티 착화 우려 여부,없음,-,S020,2,0.0,1.0,0.0,3.0,...,MAX_PER_TASK,R002,다른 배치조건도 함께 확인,미확정,작업시작전,적용,,작업 시작 직전에 확인한 현장 상태로 응답합니다. 미확인 상태를 안전한 응답으로 추...,입력,True
6,Q020,금속 칸막이·벽·천장·지붕 반대면 가연물의 열전도·복사 착화 우려 여부,없음,-,S021,2,0.0,1.0,0.0,3.0,...,MAX_PER_TASK,R002,다른 배치조건도 함께 확인,미확정,작업시작전,적용,,작업 시작 직전에 확인한 현장 상태로 응답합니다. 미확인 상태를 안전한 응답으로 추...,입력,True
7,Q025,초기소화설비 비치·접근 가능 여부,적정,-,S026,1,0.0,1.5,0.0,4.5,...,MAX_PER_TASK,R001;R014,화재위험 작업,미확정,작업시작전,적용,,작업 시작 직전에 확인한 현장 상태로 응답합니다. 미확인 상태를 안전한 응답으로 추...,입력,True
8,Q026,작업 종료 후 잔불·불티 확인 여부,None,-,S027,,NaN,NaN,0.0,0.0,...,MAX_PER_TASK,R015,,점수미산정,작업시작전,시점제외,작업 종료 후 잔불·불티 확인 항목으로 사전 점수 합산에서 제외합니다. 종료 후 확...,작업 종료 후 잔불·불티 확인 항목으로 사전 점수 합산에서 제외합니다. 종료 후 확...,미응답,False


,항목ID,확인항목,사용자응답,가중점수,관련기준ID
0,Q004,화재감시자 배치 여부,미배치,6.0,R002
1,Q003,불티 비산방지조치(용접방화포 등) 실시 여부,미실시,4.5,R001
2,Q001,주변 가연물 존재 여부,있음,1.5,R001


,항목ID,확인항목,적용상태,제외사유
0,Q001,주변 가연물 존재 여부,적용,
1,Q002,가연물과 작업점 거리,적용,
2,Q003,불티 비산방지조치(용접방화포 등) 실시 여부,적용,
3,Q004,화재감시자 배치 여부,적용,
4,Q018,용접방화포 사용 시 성능인증 제품 여부,적용,
5,Q019,작업점 하부 가연물의 불티 착화 우려 여부,적용,
6,Q020,금속 칸막이·벽·천장·지붕 반대면 가연물의 열전도·복사 착화 우려 여부,적용,
7,Q025,초기소화설비 비치·접근 가능 여부,적용,
8,Q026,작업 종료 후 잔불·불티 확인 여부,시점제외,작업 종료 후 잔불·불티 확인 항목으로 사전 점수 합산에서 제외합니다. 종료 후 확...
